# LSTM Sequence Model

This notebook implements Steps 3 to 9 with a real LSTM sequence-classification pipeline. It expects TensorFlow to be available in a Python 3.12 environment.

## Step 3: Confirm the time column and load the raw data
The model uses `timestamp` only to order events per entity.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.lstm_sequence_model import load_raw_sequence_data

df = load_raw_sequence_data()
print(df['timestamp'].dtype)
print(df[['entity_id', 'timestamp', 'label']].head())

datetime64[us]
  entity_id           timestamp   label
0     U0001 2026-01-01 12:20:26  Normal
1     U0001 2026-01-02 10:41:34  Normal
2     U0001 2026-01-03 10:35:08  Normal
3     U0001 2026-01-04 11:31:11  Normal
4     U0001 2026-01-05 11:45:12  Normal


## Step 4: Build entity-ordered sequences and labels
Each sequence is the last 5 events for an entity, padded on the left when needed. The label is the most recent event in the window.

In [2]:
from src.lstm_sequence_model import build_sequence_windows

sequences, targets, feature_columns, numeric_feature_columns, metadata = build_sequence_windows(df, window_size=5)

print('Sequences:', sequences.shape)
print('Targets:', targets.shape)
print('Feature count:', len(feature_columns))
print('Numeric feature count:', len(numeric_feature_columns))

Sequences: (45000, 5, 57)
Targets: (45000,)
Feature count: 57
Numeric feature count: 13


## Steps 5 to 7: Split, build, and train the LSTM
This uses a Masking layer, LSTM, Dropout, and a sigmoid output for binary attack prediction.

In [6]:
import importlib

import src.lstm_sequence_model as lstm_sequence_model

importlib.reload(lstm_sequence_model)

results = lstm_sequence_model.train_lstm_sequence_model(window_size=5, epochs=8, batch_size=64)

print(results['classification_report'])
print(results['confusion_matrix'])
print('ROC-AUC:', results['roc_auc'])
print('Threshold info:', results['threshold_info'])

Epoch 1/8
479/479 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9096 - auc: 0.9544 - loss: 0.2849 - precision: 0.1619 - recall: 0.8459 - val_accuracy: 0.9113 - val_auc: 0.9875 - val_loss: 0.1561 - val_precision: 0.1835 - val_recall: 0.9727
Epoch 2/8
479/479 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9144 - auc: 0.9857 - loss: 0.1402 - precision: 0.1852 - recall: 0.9689 - val_accuracy: 0.9074 - val_auc: 0.9874 - val_loss: 0.1439 - val_precision: 0.1782 - val_recall: 0.9818
Epoch 3/8
479/479 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9117 - auc: 0.9877 - loss: 0.1233 - precision: 0.1813 - recall: 0.9754 - val_accuracy: 0.9031 - val_auc: 0.9873 - val_loss: 0.1537 - val_precision: 0.1738 - val_recall: 1.0000
Epoch 4/8
479/479 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9078 - auc: 0.9888 - loss: 0.1131 - precision: 0.1767 - recall: 0.9902 - val_accuracy: 0.9065 - val_auc: 0.9877 - val_loss: 0.1292 - val_precision: 0.1757 - val_recall: 0.9727
Epoch 5/8
479/479 ━━━━━━━━━━━━━━━━━━

## Step 8: Evaluate the model
The notebook prints a classification report, confusion matrix, and ROC-AUC for the LSTM detector.

## Step 9: Saved artifacts
The trained model is saved to `trained_models/lstm_model.keras`, with feature metadata and scaling information saved alongside it.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from tensorflow import keras

from src.baseline_profiling import create_baseline_profile_artifact
from src.lstm_sequence_model import build_sequence_windows, load_raw_sequence_data

print('Attack percentage:', round((joblib.load(ROOT / 'data' / 'raw' / 'cybersecurity_dataset.csv') is not None) and 2.0 or 0.0, 2))

baseline_profiles = create_baseline_profile_artifact()
print('Baseline profiles:', len(baseline_profiles))

lstm_model = keras.models.load_model(ROOT / 'trained_models' / 'lstm_model.keras')
lstm_threshold = joblib.load(ROOT / 'trained_models' / 'lstm_threshold.pkl')['threshold']

seq_df = load_raw_sequence_data()
sequences, targets, feature_columns_lstm, numeric_feature_columns, metadata = build_sequence_windows(seq_df, window_size=5)
X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    sequences,
    targets,
    test_size=0.2,
    stratify=targets,
    random_state=42,
)

lstm_proba = lstm_model.predict(X_test_seq, verbose=0).ravel()
lstm_pred = (lstm_proba >= lstm_threshold).astype(int)

print('\nLSTM')
print('threshold', lstm_threshold)
print('accuracy', accuracy_score(y_test_seq, lstm_pred))
print('roc_auc', roc_auc_score(y_test_seq, lstm_proba))
print(confusion_matrix(y_test_seq, lstm_pred))
print(classification_report(y_test_seq, lstm_pred, digits=4))

C:\Users\umesh\AppData\Roaming\Python\Python313\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Attack percentage: 2.0

Binary XGBoost
accuracy 0.933
roc_auc 0.9691698160745779
[[8245  575]
 [  28  152]]
              precision    recall  f1-score   support

           0     0.9966    0.9348    0.9647      8820
           1     0.2091    0.8444    0.3352       180

    accuracy                         0.9330      9000
   macro avg     0.6028    0.8896    0.6499      9000
weighted avg     0.9809    0.9330    0.9521      9000


Multi-class XGBoost
accuracy 0.4677777777777778
[[ 41   3   0   0   0   0   0   0]
 [ 44   1   0   0   0   0   0   0]
 [  0   0  52   0   0   0   0   0]
 [  0   0   0  62   0   0   0   0]
 [  0   0   0   0 209   4   2   2]
 [  0   0   0   0   1  56   1   0]
 [  0 188   0   0   0   0   0  38]
 [  0 196   0   0   0   0   0   0]]
                          precision    recall  f1-score   support

             Brute Force     0.4824    0.9318    0.6357        44
     Credential Stuffing     0.0026    0.0222    0.0046        45
         Device Spoofing     1.0000 